# SN_BESP — Step 1: data splits + data index

**How to run:** attach the market-data dataset (*Add Input*), then **Run All**.

Cell 6 should print `ALL CHECKS PASSED`. Every later notebook starts with cells 1–5 of this one.

In [1]:
from pathlib import Path
BASE = Path("/kaggle/working/sn_besp")
(BASE / "config").mkdir(parents=True, exist_ok=True)
(BASE / "src").mkdir(parents=True, exist_ok=True)

In [2]:
%%writefile /kaggle/working/sn_besp/config/splits.yaml
# Single source of truth for date roles. Every script reads dates from here.
# Raw data/ stays in its original layout and is never copied into split folders:
# predict.py needs <day> + <day-1> in one directory, and split folders would break
# warm-up at every boundary.
#
# All dates are UTC days, inclusive on both ends.

version: 1
universe:
  first_day: "2025-09-15"   # first day with targets
  last_day:  "2026-09-14"   # last day with targets
  # Test = 2026-09-15 onward. Organisers hold it; we never have it.

zones:
  dev:
    start: "2025-09-15"
    end:   "2026-07-14"     # = official training period
    labels: open            # fitting + walk-forward
  selection:
    start: "2026-07-15"
    end:   "2026-08-17"     # first 34 days of official validation
    labels: open            # model comparison / hyperparameters only, never fitting
  lockbox:
    start: "2026-08-18"
    end:   "2026-09-14"     # last 28 days (4 full weeks) of official validation
    labels: sealed          # scored ONCE on the final candidate (Phase 5)

# Walk-forward inside dev: expanding window, one calendar month per test fold.
walk_forward:
  scheme: expanding
  first_test_month: "2026-01"   # gives >= 3.5 months of history to the first fold
  last_test_month:  "2026-07"   # partial month, ends at dev.end
  embargo_days: 1               # y_8s at 23:59:59.5 ends in the next day; drop 1 day between train and test

# Final submission refit uses every labelled day (rules allow validation in training).
final_refit:
  start: "2025-09-15"
  end:   "2026-09-14"

# Raw-data warm-up: the previous day's RAW market data may be read for any day,
# even across a zone boundary (predict.py gets it too). Its LABELS may not.
warmup:
  previous_day_raw_allowed_across_zones: true

# Days scored as a separate stress report (not removed from fitting by default;
# frozen/outage windows inside them are masked at row level instead).
stress_days:
  "2025-09-24": FDUSD depeg intervals
  "2025-10-10": vol shock 4-10x, USDC + FDUSD depeg
  "2025-10-11": FDUSD depeg tail
  "2026-02-23": USD1 depeg
  "2026-03-26": perp book outage ~10 min
  "2026-05-21": bbo/trades outage 25 min
  "2026-06-12": bbo/trades outage 31 min, all perp books frozen
  "2026-06-22": bbo outage 5-6 min
  "2026-07-14": bbo/trades outage 20-22 min
  "2026-07-28": BTC + PEPE-USDC perp book outage 16-35 min
  "2026-08-08": SOL spot outage 15 min

# Instruments that start later than the universe.
late_listings:
  BTCUSD1_SPOT: "2025-12-08"
  ETHUSD1_SPOT: "2025-12-15"

# Transparency record: zones whose data was already looked at in EDA.
# Descriptive EDA only; no model was fitted or tuned on the lockbox.
eda_exposure:
  - "EDA deck scored baseline / M3 / Kalman on all 62 official-validation days (incl. lockbox)"
  - "Notebook cells 17-19 read 2026-07-21..23, 07-28, 08-17..19, 09-12..14"


Writing /kaggle/working/sn_besp/config/splits.yaml


In [3]:
%%writefile /kaggle/working/sn_besp/src/splits.py
"""Date-role registry for the 500 ms perp challenge.

Usage
-----
    from splits import Splits
    s = Splits.load()                       # reads config/splits.yaml
    s.dates("dev")                          # list[str] of YYYY-MM-DD
    s.role_of("2026-08-20")                 # "lockbox"
    for fold in s.walk_forward(): ...       # Fold(name, train, test)
    s.label_days(days, purpose="fit")       # raises if a sealed day is requested

CLI
---
    python src/splits.py                    # print summary + run checks
    python src/splits.py --data-dir ./data  # also check files on disk

The lockbox is sealed: label_days() refuses lockbox dates unless the environment
variable LOCKBOX_UNLOCK holds a written reason; every unlock is appended to
lockbox_access.log next to the config.
"""
from __future__ import annotations

import argparse
import datetime as dt
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import yaml

CONFIG = Path(__file__).resolve().parents[1] / "config" / "splits.yaml"
PERPS = ("BTCUSDT_PERP", "ETHUSDT_PERP", "SOLUSDT_PERP", "1000PEPEUSDT_PERP")


def _d(s: str) -> dt.date:
    return dt.date.fromisoformat(s)


def _span(a: dt.date, b: dt.date) -> list[str]:
    return [(a + dt.timedelta(n)).isoformat() for n in range((b - a).days + 1)]


@dataclass(frozen=True)
class Fold:
    name: str
    train: tuple[str, ...]
    test: tuple[str, ...]


class Splits:
    def __init__(self, cfg: dict, path: Path):
        self.cfg, self.path = cfg, path
        self.zones = {
            z: _span(_d(v["start"]), _d(v["end"])) for z, v in cfg["zones"].items()
        }
        self._role = {d: z for z, ds in self.zones.items() for d in ds}

    @classmethod
    def load(cls, path: Path | str = CONFIG) -> "Splits":
        path = Path(path)
        return cls(yaml.safe_load(path.read_text()), path)

    # ---- lookups -------------------------------------------------------
    def dates(self, zone: str) -> list[str]:
        return list(self.zones[zone])

    def role_of(self, day: str) -> str:
        if day > self.cfg["universe"]["last_day"]:
            return "test"
        return self._role.get(day, "outside")

    @staticmethod
    def warmup_day(day: str) -> str:
        """Previous UTC day: its raw market data may be read for features."""
        return (_d(day) - dt.timedelta(1)).isoformat()

    def stress_days(self) -> list[str]:
        return sorted(self.cfg.get("stress_days", {}))

    def final_refit_days(self) -> list[str]:
        f = self.cfg["final_refit"]
        return _span(_d(f["start"]), _d(f["end"]))

    # ---- label access guard -------------------------------------------
    def label_days(self, days, purpose: str) -> list[str]:
        """Return days whose labels may be used; refuse sealed zones."""
        days = list(days)
        sealed = [d for d in days
                  if self.cfg["zones"].get(self.role_of(d), {}).get("labels") == "sealed"]
        if any(self.role_of(d) == "test" for d in days):
            raise PermissionError("test-period labels do not exist")
        if sealed:
            reason = os.environ.get("LOCKBOX_UNLOCK", "").strip()
            if not reason:
                raise PermissionError(
                    f"{len(sealed)} lockbox day(s) requested for '{purpose}'. "
                    "Set LOCKBOX_UNLOCK='<reason>' only for the single final score.")
            log = self.path.parent / "lockbox_access.log"
            with log.open("a") as fh:
                fh.write(f"{dt.datetime.now(dt.timezone.utc).isoformat()}\t"
                         f"{purpose}\t{len(sealed)} days\t{reason}\n")
        return days

    # ---- walk-forward -------------------------------------------------
    def walk_forward(self) -> list[Fold]:
        wf = self.cfg["walk_forward"]
        dev = self.zones["dev"]
        emb = int(wf["embargo_days"])
        y0, m0 = map(int, wf["first_test_month"].split("-"))
        y1, m1 = map(int, wf["last_test_month"].split("-"))
        folds, y, m = [], y0, m0
        while (y, m) <= (y1, m1):
            test = [d for d in dev if _d(d).year == y and _d(d).month == m]
            if test:
                cutoff = _d(test[0]) - dt.timedelta(emb)
                train = [d for d in dev if _d(d) < cutoff]
                folds.append(Fold(f"{y}-{m:02d}", tuple(train), tuple(test)))
            y, m = (y + 1, 1) if m == 12 else (y, m + 1)
        return folds

    # ---- checks -------------------------------------------------------
    def check(self, data_dir: Path | None = None) -> list[str]:
        errs = []
        u0, u1 = _d(self.cfg["universe"]["first_day"]), _d(self.cfg["universe"]["last_day"])
        universe = _span(u0, u1)
        all_days = [d for ds in self.zones.values() for d in ds]
        if len(all_days) != len(set(all_days)):
            errs.append("zones overlap")
        if sorted(all_days) != universe:
            errs.append("zones do not tile the labelled universe exactly")
        order = sorted(self.zones, key=lambda z: self.zones[z][0])
        if order != ["dev", "selection", "lockbox"]:
            errs.append(f"zone order is {order}, expected dev < selection < lockbox")
        if self.final_refit_days() != universe:
            errs.append("final_refit must cover the full labelled universe")
        emb = int(self.cfg["walk_forward"]["embargo_days"])
        for f in self.walk_forward():
            if any(self.role_of(d) != "dev" for d in f.train + f.test):
                errs.append(f"fold {f.name} leaves dev")
            if f.train and (_d(f.test[0]) - _d(f.train[-1])).days <= emb:
                errs.append(f"fold {f.name} violates embargo")
        for d in self.stress_days():
            if self.role_of(d) == "outside":
                errs.append(f"stress day {d} outside universe")
        if data_dir is not None:
            errs += self._check_files(Path(data_dir), universe)
        return errs

    def _check_files(self, root: Path, universe: list[str]) -> list[str]:
        """root may be a data/ folder or any folder above split data/ folders."""
        from data_index import DataIndex
        return DataIndex.build(root).problems(universe)

    def summary(self) -> str:
        rows = [f"{'zone':<10}{'start':>12}{'end':>12}{'days':>6}  labels"]
        for z, ds in self.zones.items():
            rows.append(f"{z:<10}{ds[0]:>12}{ds[-1]:>12}{len(ds):>6}  "
                        f"{self.cfg['zones'][z]['labels']}")
        rows.append("\nwalk-forward folds (expanding, "
                    f"embargo {self.cfg['walk_forward']['embargo_days']}d):")
        for f in self.walk_forward():
            rows.append(f"  {f.name}: train {f.train[0]}..{f.train[-1]} "
                        f"({len(f.train)}d)  test {f.test[0]}..{f.test[-1]} ({len(f.test)}d)")
        st = self.stress_days()
        rows.append(f"\nstress days: {len(st)} "
                    f"({', '.join(f'{d}:{self.role_of(d)}' for d in st)})")
        return "\n".join(rows)


def main(argv=None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=str(CONFIG))
    ap.add_argument("--data-dir", default=None)
    a = ap.parse_args(argv)
    s = Splits.load(a.config)
    print(s.summary())
    errs = s.check(a.data_dir)
    print("\nchecks:", "OK" if not errs else "FAILED")
    for e in errs:
        print("  -", e)
    return 1 if errs else 0


if __name__ == "__main__":
    sys.exit(main())


Writing /kaggle/working/sn_besp/src/splits.py


In [4]:
%%writefile /kaggle/working/sn_besp/src/data_index.py
"""Find data files wherever they live under the attached inputs.

Works for one combined data/ folder and for a dataset split into per-instrument
folders (e.g. sn-bes/data-BTCUSDT_PERP/data/...), as on Kaggle.

    from data_index import DataIndex
    IX = DataIndex.build("/kaggle/input")
    IX.file("book", "BTCUSDT_PERP", "2026-01-05")   # Path or None
    IX.stable("2026-01-05")                          # Path or None
    IX.meta("target_clip.parquet")                   # Path or None
    print(IX.summary())
"""
from __future__ import annotations

import glob
from pathlib import Path

STREAMS = ("book", "bbo", "trades", "targets")
INSTRUMENTS = (
    "BTCUSDT_PERP", "ETHUSDT_PERP", "SOLUSDT_PERP", "1000PEPEUSDT_PERP",
    "1000PEPEUSDC_PERP",
    "BTCUSDT_SPOT", "BTCUSDC_SPOT", "BTCFDUSD_SPOT", "BTCUSD1_SPOT",
    "ETHUSDT_SPOT", "ETHUSDC_SPOT", "ETHFDUSD_SPOT", "ETHUSD1_SPOT",
    "SOLUSDT_SPOT", "SOLUSDC_SPOT", "SOLFDUSD_SPOT",
    "PEPEUSDT_SPOT",
)
PERPS = INSTRUMENTS[:4]


class DataIndex:
    def __init__(self, folders: dict[tuple[str, str], Path]):
        self.folders = folders

    @classmethod
    def build(cls, root: str | Path = "/kaggle/input") -> "DataIndex":
        root = str(root).rstrip("/")
        folders: dict[tuple[str, str], Path] = {}

        def add(key, path):
            if key in folders and folders[key] != path:
                raise RuntimeError(f"{key} found twice: {folders[key]} and {path}")
            folders[key] = path

        # root itself may be a data/ folder, or contain data/ folders anywhere below
        for base in [root] + sorted(glob.glob(f"{root}/**/data", recursive=True)):
            b = Path(base)
            for s in STREAMS:
                for p in sorted(b.glob(f"{s}/*/")):
                    if p.is_dir():
                        add((s, p.name), p)
            for s in ("stable", "meta"):
                if (b / s).is_dir():
                    add((s, ""), b / s)
        return cls(folders)

    # ---- lookups -------------------------------------------------------
    def folder(self, stream: str, instrument: str = "") -> Path | None:
        return self.folders.get((stream, instrument))

    def file(self, stream: str, instrument: str, day: str) -> Path | None:
        d = self.folder(stream, instrument)
        if d is None:
            return None
        f = d / f"{day}.parquet"
        return f if f.exists() else None

    def stable(self, day: str) -> Path | None:
        d = self.folder("stable")
        f = d / f"{day}.parquet" if d else None
        return f if f is not None and f.exists() else None

    def meta(self, name: str) -> Path | None:
        d = self.folder("meta")
        f = d / name if d else None
        return f if f is not None and f.exists() else None

    def days(self, stream: str, instrument: str = "") -> set[str]:
        d = self.folder(stream, instrument)
        return {p.stem for p in d.glob("*.parquet")} if d else set()

    # ---- checks --------------------------------------------------------
    def problems(self, universe: list[str]) -> list[str]:
        errs = []
        for s in ("book", "bbo", "trades"):
            for inst in INSTRUMENTS:
                if self.folder(s, inst) is None:
                    errs.append(f"{s}/{inst}: folder not found")
        for inst in PERPS:
            if self.folder("targets", inst) is None:
                errs.append(f"targets/{inst}: folder not found")
                continue
            have = self.days("targets", inst)
            missing = sorted(set(universe) - have)
            extra = sorted(have - set(universe))
            if missing:
                errs.append(f"targets/{inst}: {len(missing)} days missing, e.g. {missing[:3]}")
            if extra:
                errs.append(f"targets/{inst}: days outside universe, e.g. {extra[:3]}")
        if self.folder("stable") is None:
            errs.append("stable/: folder not found")
        if self.meta("target_clip.parquet") is None:
            errs.append("meta/target_clip.parquet: not found")
        return errs

    def summary(self) -> str:
        rows = []
        for s in STREAMS:
            n = sum(1 for (st, _) in self.folders if st == s)
            rows.append(f"{s:8} {n:3} instrument folders")
        for s in ("stable", "meta"):
            rows.append(f"{s:8} {'found' if self.folder(s) else 'MISSING'}")
        return "\n".join(rows)


Writing /kaggle/working/sn_besp/src/data_index.py


In [5]:
import sys
sys.path.insert(0, "/kaggle/working/sn_besp/src")
from splits import Splits
from data_index import DataIndex

S  = Splits.load()                       # date roles
IX = DataIndex.build("/kaggle/input")    # where every file lives
print(S.summary()); print(); print(IX.summary())

zone             start         end  days  labels
dev         2025-09-15  2026-07-14   303  open
selection   2026-07-15  2026-08-17    34  open
lockbox     2026-08-18  2026-09-14    28  sealed

walk-forward folds (expanding, embargo 1d):
  2026-01: train 2025-09-15..2025-12-30 (107d)  test 2026-01-01..2026-01-31 (31d)
  2026-02: train 2025-09-15..2026-01-30 (138d)  test 2026-02-01..2026-02-28 (28d)
  2026-03: train 2025-09-15..2026-02-27 (166d)  test 2026-03-01..2026-03-31 (31d)
  2026-04: train 2025-09-15..2026-03-30 (197d)  test 2026-04-01..2026-04-30 (30d)
  2026-05: train 2025-09-15..2026-04-29 (227d)  test 2026-05-01..2026-05-31 (31d)
  2026-06: train 2025-09-15..2026-05-30 (258d)  test 2026-06-01..2026-06-30 (30d)
  2026-07: train 2025-09-15..2026-06-29 (288d)  test 2026-07-01..2026-07-14 (14d)

stress days: 11 (2025-09-24:dev, 2025-10-10:dev, 2025-10-11:dev, 2026-02-23:dev, 2026-03-26:dev, 2026-05-21:dev, 2026-06-12:dev, 2026-06-22:dev, 2026-07-14:dev, 2026-07-28:selection, 2026-

In [6]:
errs = IX.problems(S.final_refit_days())
print("ALL CHECKS PASSED" if not errs else "PROBLEMS:\n  - " + "\n  - ".join(errs))

ALL CHECKS PASSED


In [7]:
# How later notebooks get a file
day = S.dates("dev")[100]
print(day, "->", IX.file("book", "BTCUSDT_PERP", day))
print("warm-up day:", S.warmup_day(day), "->", IX.file("book", "BTCUSDT_PERP", S.warmup_day(day)))
print("stable     :", IX.stable(day))
print("clip table :", IX.meta("target_clip.parquet"))
try:
    S.label_days(S.dates("lockbox")[:1], purpose="guard test")
except PermissionError as e:
    print("\nlockbox guard works ->", e)

2025-12-24 -> /kaggle/input/datasets/sumalya32/sn-bes/data-BTCUSDT_PERP/data/book/BTCUSDT_PERP/2025-12-24.parquet
warm-up day: 2025-12-23 -> /kaggle/input/datasets/sumalya32/sn-bes/data-BTCUSDT_PERP/data/book/BTCUSDT_PERP/2025-12-23.parquet
stable     : /kaggle/input/datasets/sumalya32/sn-bes/data-meta/data/stable/2025-12-24.parquet
clip table : /kaggle/input/datasets/sumalya32/sn-bes/data-meta/data/meta/target_clip.parquet

lockbox guard works -> 1 lockbox day(s) requested for 'guard test'. Set LOCKBOX_UNLOCK='<reason>' only for the single final score.
